# MiniCPM5-1B × Muse Spark 蒸馏：QLoRA 微调（Kaggle / Colab 双兼容）

基座 `openbmb/MiniCPM5-1B-Base`（标准 Llama 架构，Apache-2.0）＋本仓库 173 条蒸馏数据，Unsloth QLoRA r16，T4 免费卡约 30 分钟跑完。

**开始前（三件事，你来点）：**
1. Kaggle：右侧 Settings → Accelerator 选 **GPU T4 x2**（每周 30h 免费），Internet 保持 **On**；或 Colab：运行时 → 更改运行时类型 → **T4 GPU**。
2. 把下面 `REPO` 改成你 fork 后的地址（或直接用原仓，公开可读）。
3. 想把 adapter 推到 HuggingFace：在 Kaggle Secrets / Colab 左侧钥匙图标里加 `HF_TOKEN`（huggingface.co/settings/tokens 拿，只读+写模型权限）。没有 token 也能训，只是保存在本地。

如果 Unsloth 在某天对这个架构抽风，用同目录 `train_fallback.py`（纯 transformers+PEFT，慢一点但稳）。

In [ ]:
import torch, os
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("显存 GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
assert torch.cuda.is_available(), "没检测到 GPU！去检查 Accelerator/运行时类型是不是选了 GPU。"

In [ ]:
# 装依赖（约 3-5 分钟）。Kaggle 装完如提示重启内核就重启，再从下一个 cell 继续。
!pip install -q unsloth datasets trl peft accelerate bitsandbytes

In [ ]:
REPO = "https://github.com/ice-wocker/spark-distill-1b.git"  # ← 改成你 fork 的地址也行
!rm -rf spark-distill-1b && git clone --depth 1 $REPO
import json
rows = [json.loads(l) for l in open("spark-distill-1b/data/train.jsonl", encoding="utf-8")]
print("训练样本:", len(rows))
print("示例:", rows[0]["instruction"][:40], "→", rows[0]["output"][:40])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="openbmb/MiniCPM5-1B-Base",
    max_seq_length=2048,   # 数据都是短问答，2048 够了，还省显存；131k 原生上下文推理时再用
    dtype=None,            # 自动：T4 用 fp16，有 bf16 就用 bf16
    load_in_4bit=True,     # QLoRA：1B 四比特，T4 上不到 4GB
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0, bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42, max_seq_length=2048,
)
print("LoRA 可训练参数:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# 用 tokenizer 自带的 chat_template 拼训练文本（MiniCPM5 的 Think/No-Think 模板以官方为准，我们不手写）。
EOS = tokenizer.eos_token

def to_text(r):
    user = r["instruction"] + ("\n" + r["input"] if r.get("input") else "")
    msgs = [{"role": "user", "content": user},
            {"role": "assistant", "content": r["output"]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False) + EOS

texts = [to_text(r) for r in rows]
print("平均长度:", sum(len(tokenizer.encode(t)) for t in texts) // len(texts), "tokens")
print(texts[0][:300])

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

ds = Dataset.from_dict({"text": texts})

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=2048, dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,  # 有效 batch 8
        num_train_epochs=3, learning_rate=2e-4, lr_scheduler_type="cosine",
        warmup_steps=10, logging_steps=10, save_steps=50, save_total_limit=2,
        output_dir="spark-1b-lora", optim="adamw_8bit", seed=42,
        report_to="none",
    ),
)
trainer.train()  # 173 条 x 3 epoch，T4 上约 20-40 分钟

In [ ]:
# 跑前抽查：同一个问题看手感（别指望 173 条就脱胎换骨，看“风格对不对”）
FastLanguageModel.for_inference(model)
for q in ["解释一下 Python 的 GIL", "写一个判断质数的函数", "用三句话介绍杭州"]:
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=256, use_cache=True)
    print("Q:", q)
    print("A:", tokenizer.batch_decode(out[:, inp.input_ids.shape[1]:])[0][:400])
    print("-" * 40)

In [ ]:
# 保存 adapter（小文件，几十 MB）。填了 HF_TOKEN 就推送，没填就只存本地下载。
model.save_pretrained("spark-1b-lora")
tokenizer.save_pretrained("spark-1b-lora")
print("本地已保存 spark-1b-lora/")

import os
tok = os.environ.get("HF_TOKEN", "")
if tok:
    from huggingface_hub import login
    login(token=tok)
    model.push_to_hub("你的用户名/spark-1b-lora")  # ← 改成你的名字
    tokenizer.push_to_hub("你的用户名/spark-1b-lora")
    print("已推送到 Hub")
else:
    print("没检测到 HF_TOKEN，跳过推送（右侧 Download 可把 spark-1b-lora 拉下来）")

## 下一步

1. **评测**：把 `train/eval.py` 传上来（或贴进去跑），在 eval 集上打分 + 生成抽查。
2. **合并且转 GGUF**：`export/` 下有合并 LoRA + 转 Q4_K_M 的脚本，转完就能塞进 Termux/llama.cpp 跑。
3. **扩量**：173 条只是种子。`scripts/expand.py` 能批量生成新 instruction，你拿去找老师模型（比如继续找我）要答案，加到 `data/seed/`，CI 校验，下一轮训 1000+ 条。